# Course 1 — Classification + KNN

What does a classifier output? How do we measure it? KNN is the
simplest learner and the perfect place to meet decision boundaries,
confusion matrices, and ROC curves.

**Sections**
1. What a classifier outputs (0:00–0:30)
2. KNN — the no-training baseline (0:30–1:00)
3. Class imbalance and the majority-class trap (1:00–1:30)

In [ ]:
import sys, pathlib
p = pathlib.Path.cwd()
while not (p / 'pyproject.toml').exists() and p != p.parent:
    p = p.parent
if str(p) not in sys.path:
    sys.path.insert(0, str(p))
from shared.data_utils import load_dataset
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import (confusion_matrix, classification_report,
                              roc_curve, auc, ConfusionMatrixDisplay)
rng = np.random.default_rng(0)
default = load_dataset('default')
print(default.head())
print('default rate:', (default['default'] == 'Yes').mean().round(4))


## 1. What a classifier outputs

A classifier maps an input vector to a discrete label, but most modern classifiers first produce a **probability** and then threshold it (default 0.5). The **decision boundary** is the set of points where P(Y=1|X) = 0.5 — everything on one side is labelled positive, everything on the other negative. Shifting the threshold trades precision for recall without retraining.

### Look at the data first

In [ ]:
d = default.copy()
d['y'] = (d['default'] == 'Yes').astype(int)
fig, ax = plt.subplots(figsize=(6, 4))
ax.scatter(d.loc[d.y==0, 'balance'], d.loc[d.y==0, 'income'], s=3, alpha=0.3, label='No')
ax.scatter(d.loc[d.y==1, 'balance'], d.loc[d.y==1, 'income'], s=10, color='C3', label='Yes')
ax.set_xlabel('balance'); ax.set_ylabel('income'); ax.legend()
ax.set_title('Default — balance is the separating feature'); plt.show()


### Confusion matrix and metrics

Every binary classification result can be summarised in a 2×2 **confusion matrix**:

| | Predicted Negative | Predicted Positive |
|---|---|---|
| **Actual Negative** | TN — true negatives | FP — false positives (Type I error) |
| **Actual Positive** | FN — false negatives (Type II error) | TP — true positives |

From these four numbers every standard metric is derived.


The simplest possible classifier: 'predict default if balance > 1700'.
Score it like any other model.

In [ ]:
y = d['y'].to_numpy()
y_hat = (d['balance'] > 1700).astype(int).to_numpy()
# rows = actual, cols = predicted; off-diagonal = errors
cm = confusion_matrix(y, y_hat)
print('confusion matrix [TN FP / FN TP]:'); print(cm)
print()
print(classification_report(y, y_hat, target_names=['No', 'Yes']))

#### The core metrics

**Accuracy** = (TP + TN) / (TP + TN + FP + FN)  
Fraction of all predictions that are correct.  
> Use only when classes are roughly balanced. On a dataset that is 97% negative, predicting *always negative* achieves 97% accuracy while catching zero positives — accuracy lies.

---

**Precision** = TP / (TP + FP)  
Of every positive prediction, what fraction is actually positive.  
> Use when **false positives are costly**: spam filter (a real email in spam is bad), recommendation systems (irrelevant results hurt trust).

**Recall (Sensitivity, TPR)** = TP / (TP + FN)  
Of every actual positive, what fraction did we catch.  
> Use when **false negatives are costly**: cancer screening (missing a tumour is worse than a false alarm), fraud detection.

**Specificity (TNR)** = TN / (TN + FP)  
Of every actual negative, what fraction did we correctly call negative.  

**FPR** = FP / (FP + TN) = 1 − Specificity — used on the x-axis of the ROC curve.

---

#### F-scores: trading off precision vs recall

**F1** = 2 · Precision · Recall / (Precision + Recall)  
The harmonic mean of precision and recall. Harmonic mean penalises extreme imbalance — an F1 of 0.9 requires *both* precision and recall to be high.
> Use when false positives and false negatives are equally costly.

**Fβ** = (1 + β²) · Precision · Recall / (β² · Precision + Recall)  
Generalises F1 by weighting recall β times more than precision.

| β | Emphasis | Typical use case |
|---|---|---|
| β < 1 (e.g. 0.5) | Precision matters more | Spam filter — minimise false positives |
| β = 1 | Balanced | General purpose |
| β > 1 (e.g. 2) | Recall matters more | Cancer screening — minimise false negatives |

So β is literally the ratio: *how many times more important is recall than precision?*

---

#### Multi-class averaging strategies

When there are K > 2 classes, sklearn's `classification_report` can average per-class scores in three ways:

| Average | Formula | When to use |
|---|---|---|
| **macro** | Simple mean across all K classes | Every class equally important, regardless of size |
| **weighted** | Mean weighted by class support (n_k / n) | Class sizes differ; you care more about frequent classes |
| **micro** | Pool TP/FP/FN across all classes, then compute | Dominated by large classes; equivalent to accuracy when there are no multi-label predictions |
| **samples** | Per-sample average | Multi-label classification only |

Rule of thumb: use **macro** for equal-class importance (e.g. rare disease detection), **weighted** when class frequencies reflect real-world distribution.

---

**AUC-ROC** = area under the Receiver Operating Characteristic curve.  
Plots TPR vs FPR as the classification threshold varies from 1 → 0.  
AUC = P(score(positive) > score(negative)) — the probability a random positive ranks above a random negative.  
> Threshold-free. AUC = 0.5 is random guessing; AUC = 1.0 is perfect. Prefer over accuracy on imbalanced data.

**AUC-PR** (Precision-Recall curve) — plots precision vs recall across thresholds.  
> Even better than AUC-ROC when the positive class is very rare (< 5%).


In [ ]:
from sklearn.metrics import (classification_report, f1_score,
                              fbeta_score, precision_recall_curve,
                              average_precision_score)
import numpy as np

# Re-use y and y_hat from the confusion matrix cell above
print('=== Fβ scores ===')
for beta in [0.5, 1.0, 2.0]:
    score = fbeta_score(y, y_hat, beta=beta, pos_label=1, zero_division=0)
    print(f'  F{beta:.1f}: {score:.4f}  (beta={beta:.1f} weights recall {beta:.1f}x more than precision)')

print()
print('=== Multi-class averaging (macro / weighted / micro) ===')
# Simulate a 3-class problem for illustration
rng = np.random.default_rng(0)
y3      = rng.integers(0, 3, 300)
y3_hat  = rng.integers(0, 3, 300)
for avg in ['macro', 'weighted', 'micro']:
    score = f1_score(y3, y3_hat, average=avg)
    print(f'  {avg:10s} F1: {score:.4f}')
print()
print('Full report (includes per-class + macro/weighted):')
print(classification_report(y3, y3_hat))


#### Visualising the precision–recall tradeoff

Moving the decision threshold trades precision for recall. The plot below shows how each metric changes as we sweep the threshold from 0 → 1 on the Default dataset.


In [ ]:
from sklearn.metrics import precision_recall_curve, average_precision_score
import numpy as np, matplotlib.pyplot as plt

# Use a logistic regression to get calibrated probabilities
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

X_lr = d[['balance','income']].to_numpy()
y_lr = d['y'].to_numpy()
Xtr_lr, Xte_lr, ytr_lr, yte_lr = train_test_split(X_lr, y_lr, test_size=0.3, random_state=0, stratify=y_lr)
sc_lr = StandardScaler().fit(Xtr_lr)
lr_pr = LogisticRegression(max_iter=1000).fit(sc_lr.transform(Xtr_lr), ytr_lr)
proba_pr = lr_pr.predict_proba(sc_lr.transform(Xte_lr))[:,1]

thresholds = np.linspace(0.01, 0.99, 200)
precisions, recalls, f1s = [], [], []
from sklearn.metrics import precision_score, recall_score, f1_score
for t in thresholds:
    yh = (proba_pr >= t).astype(int)
    precisions.append(precision_score(yte_lr, yh, zero_division=0))
    recalls.append(recall_score(yte_lr, yh, zero_division=0))
    f1s.append(f1_score(yte_lr, yh, zero_division=0))

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Left: metrics vs threshold
ax = axes[0]
ax.plot(thresholds, precisions, label='Precision', color='steelblue')
ax.plot(thresholds, recalls,    label='Recall',    color='tomato')
ax.plot(thresholds, f1s,        label='F1',        color='seagreen', linestyle='--')
ax.axvline(0.5, color='gray', linestyle=':', label='threshold=0.5')
best_t = thresholds[np.argmax(f1s)]
ax.axvline(best_t, color='seagreen', linestyle=':', label=f'best F1 (t={best_t:.2f})')
ax.set_xlabel('Decision threshold'); ax.set_ylabel('Score')
ax.set_title('Metrics vs Threshold — Default')
ax.legend(fontsize=8); ax.set_ylim(0, 1)

# Right: precision-recall curve
ax2 = axes[1]
pr_p, pr_r, _ = precision_recall_curve(yte_lr, proba_pr)
ap = average_precision_score(yte_lr, proba_pr)
ax2.plot(pr_r, pr_p, color='steelblue', lw=2)
ax2.axhline(yte_lr.mean(), color='gray', linestyle='--', label=f'baseline (= positive rate {yte_lr.mean():.2f})')
ax2.set_xlabel('Recall'); ax2.set_ylabel('Precision')
ax2.set_title(f'Precision–Recall Curve  (AP = {ap:.3f})')
ax2.legend(fontsize=8); ax2.set_xlim(0,1); ax2.set_ylim(0,1)

plt.tight_layout()
plt.show()
print(f'Best F1 threshold: {best_t:.3f}  F1={max(f1s):.3f}')


#### Fβ — how β shifts emphasis between precision and recall

The plot below shows the Fβ score surface as a function of precision and recall for three values of β.


In [ ]:
import numpy as np, matplotlib.pyplot as plt
from sklearn.metrics import fbeta_score

# Surface plot: Fbeta as a function of precision x recall
P = np.linspace(0.01, 1, 100)
R = np.linspace(0.01, 1, 100)
PP, RR = np.meshgrid(P, R)

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, beta, title in zip(axes,
                            [0.5, 1.0, 2.0],
                            ['F0.5  (precision matters more)',
                             'F1    (balanced)',
                             'F2    (recall matters more)']):
    Fb = (1 + beta**2) * PP * RR / (beta**2 * PP + RR)
    cf = ax.contourf(PP, RR, Fb, levels=20, cmap='RdYlGn')
    plt.colorbar(cf, ax=ax)
    ax.set_xlabel('Precision'); ax.set_ylabel('Recall')
    ax.set_title(title)
    # Mark a few iso-Fbeta lines
    ax.contour(PP, RR, Fb, levels=[0.3, 0.5, 0.7, 0.9], colors='black', linewidths=0.8, linestyles='--')

plt.suptitle('Fβ score as a function of Precision and Recall', y=1.02)
plt.tight_layout()
plt.show()
print('Iso-lines shift toward recall axis as β increases — high recall becomes the priority.')


#### Multi-class averaging strategies — macro vs weighted vs micro

The three averaging strategies give different answers when class sizes differ. The example below uses a synthetic 3-class problem where one class is rare.


In [ ]:
import numpy as np, matplotlib.pyplot as plt
from sklearn.metrics import classification_report, ConfusionMatrixDisplay, confusion_matrix
from sklearn.linear_model import LogisticRegression
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Imbalanced 3-class dataset: class 0 (50%), class 1 (45%), class 2 (5%)
X3, y3 = make_classification(
    n_samples=1000, n_features=10, n_informative=5,
    n_classes=3, n_clusters_per_class=1,
    weights=[0.50, 0.45, 0.05], random_state=42)

Xtr3, Xte3, ytr3, yte3 = train_test_split(X3, y3, test_size=0.3, random_state=0, stratify=y3)
sc3 = StandardScaler().fit(Xtr3)
lr3 = LogisticRegression(max_iter=1000, random_state=0).fit(sc3.transform(Xtr3), ytr3)
yhat3 = lr3.predict(sc3.transform(Xte3))

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Left: confusion matrix
ConfusionMatrixDisplay.from_predictions(
    yte3, yhat3,
    display_labels=['Class 0 (50%)', 'Class 1 (45%)', 'Class 2 (5%)'],
    ax=axes[0], colorbar=False)
axes[0].set_title('Confusion Matrix')

# Right: bar chart of F1 per class + three averages
from sklearn.metrics import f1_score
per_class = f1_score(yte3, yhat3, average=None)
macro_f1    = f1_score(yte3, yhat3, average='macro')
weighted_f1 = f1_score(yte3, yhat3, average='weighted')
micro_f1    = f1_score(yte3, yhat3, average='micro')

ax2 = axes[1]
bars = ax2.bar(['Class 0\n(50%)', 'Class 1\n(45%)', 'Class 2\n(5%)'],
               per_class, color=['steelblue','steelblue','tomato'], alpha=0.8)
ax2.axhline(macro_f1,    color='seagreen',  linestyle='--', lw=2, label=f'Macro F1 = {macro_f1:.3f}')
ax2.axhline(weighted_f1, color='orange',    linestyle='-.',  lw=2, label=f'Weighted F1 = {weighted_f1:.3f}')
ax2.axhline(micro_f1,    color='purple',    linestyle=':',  lw=2, label=f'Micro F1 = {micro_f1:.3f}')
ax2.set_ylim(0, 1.1); ax2.set_ylabel('F1')
ax2.set_title('Per-class F1 + Three Averages')
ax2.legend(fontsize=8)

plt.tight_layout(); plt.show()
print('Macro drags down because of Class 2. Weighted barely notices it — 5% of samples.')
print(classification_report(yte3, yhat3, target_names=['Class 0','Class 1','Class 2']))


#### AUC-ROC vs AUC-PR — which to use on imbalanced data?

AUC-ROC can look optimistic on severely imbalanced data because FPR has many TN in the denominator. AUC-PR (Average Precision) is more honest about minority class performance.


In [ ]:
import numpy as np, matplotlib.pyplot as plt
from sklearn.metrics import (roc_curve, roc_auc_score,
                              precision_recall_curve, average_precision_score)
from sklearn.linear_model import LogisticRegression
from sklearn.dummy import DummyClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

X_c = d[['balance','income']].to_numpy()
y_c = d['y'].to_numpy()
Xt, Xe, yt, ye = train_test_split(X_c, y_c, test_size=0.3, random_state=0, stratify=y_c)
sc = StandardScaler().fit(Xt)

models = {
    'Logistic Regression': LogisticRegression(max_iter=1000).fit(sc.transform(Xt), yt),
    'Random baseline': DummyClassifier(strategy='stratified', random_state=0).fit(Xt, yt),
}

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
colors = ['steelblue', 'tomato']

for (name, model), color in zip(models.items(), colors):
    prob = model.predict_proba(sc.transform(Xe))[:,1]

    # ROC
    fpr, tpr, _ = roc_curve(ye, prob)
    auc = roc_auc_score(ye, prob)
    axes[0].plot(fpr, tpr, color=color, lw=2, label=f'{name}  AUC={auc:.3f}')

    # PR
    p, r, _ = precision_recall_curve(ye, prob)
    ap = average_precision_score(ye, prob)
    axes[1].plot(r, p, color=color, lw=2, label=f'{name}  AP={ap:.3f}')

# ROC diagonal
axes[0].plot([0,1],[0,1],'k--',lw=1,label='Random (AUC=0.5)')
axes[0].set_xlabel('FPR (False Positive Rate)'); axes[0].set_ylabel('TPR (Recall)')
axes[0].set_title('ROC Curve'); axes[0].legend(fontsize=8)

# PR baseline
axes[1].axhline(ye.mean(), color='gray', linestyle='--', lw=1,
                label=f'Baseline = positive rate ({ye.mean():.3f})')
axes[1].set_xlabel('Recall'); axes[1].set_ylabel('Precision')
axes[1].set_title('Precision–Recall Curve')
axes[1].set_xlim(0,1); axes[1].set_ylim(0,1)
axes[1].legend(fontsize=8)

plt.tight_layout(); plt.show()
print(f'Positive class rate: {ye.mean():.3f} — the PR baseline is {ye.mean():.3f}')
print('Note: ROC looks great (AUC≈0.94). PR is harder — AP reflects the minority-class challenge.')


### ROC and AUC

In [ ]:
from sklearn.linear_model import LogisticRegression
X = d[['balance', 'income']].to_numpy()
# stratify=y preserves class ratio in both train and test splits
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.3, random_state=0, stratify=y)
lr = LogisticRegression(max_iter=2000).fit(Xtr, ytr)
scores = lr.predict_proba(Xte)[:, 1]
fpr, tpr, _ = roc_curve(yte, scores)
fig, ax = plt.subplots(figsize=(5, 5))
# ROC plots TPR vs FPR across all thresholds — AUC summarises the whole curve
ax.plot(fpr, tpr); ax.plot([0, 1], [0, 1], 'k--', alpha=0.4)
ax.set_xlabel('FPR'); ax.set_ylabel('TPR')
ax.set_title(f'ROC — AUC = {auc(fpr, tpr):.4f}'); plt.show()

## 2. KNN — the no-training baseline

KNN stores the entire training set and classifies a new point by majority vote among its k nearest neighbours — there is no explicit training step. The choice of k directly controls the **bias-variance trade-off**: small k (e.g. k=1) produces a jagged, low-bias but high-variance boundary that memorises noise; large k smooths the boundary at the cost of higher bias. Scaling is essential because KNN uses Euclidean distance and large-magnitude features otherwise dominate the vote.

### Why scale before KNN

In [ ]:
print('balance range:', d['balance'].min(), '...', d['balance'].max())
print('income range :', d['income'].min(), '...', d['income'].max())


Income dwarfs balance numerically — without scaling, KNN's 'nearest'
is decided by income alone.

In [ ]:
# KNN is distance-based — unscaled features dominate by magnitude
pipe = Pipeline([('s', StandardScaler()), ('knn', KNeighborsClassifier(n_neighbors=5))])
pipe.fit(Xtr, ytr)
print(f'KNN(k=5) test accuracy = {pipe.score(Xte, yte):.4f}')

### How does k change the decision boundary?

In [ ]:
# Subsample for plotting speed
rs = rng.choice(len(Xtr), 1500, replace=False)
Xs, ys = Xtr[rs], ytr[rs]
x_min, x_max = Xs[:, 0].min() - 100, Xs[:, 0].max() + 100
y_min, y_max = Xs[:, 1].min() - 1000, Xs[:, 1].max() + 1000
xx, yy = np.meshgrid(np.linspace(x_min, x_max, 120), np.linspace(y_min, y_max, 120))
fig, axes = plt.subplots(1, 3, figsize=(13, 4), sharey=True)
for ax, k in zip(axes, [1, 5, 25]):
    pipe = Pipeline([('s', StandardScaler()), ('knn', KNeighborsClassifier(k))])
    pipe.fit(Xs, ys)
    Z = pipe.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
    ax.contourf(xx, yy, Z, alpha=0.2, levels=[-0.5, 0.5, 1.5])
    ax.scatter(Xs[ys==0, 0], Xs[ys==0, 1], s=3, alpha=0.3)
    ax.scatter(Xs[ys==1, 0], Xs[ys==1, 1], s=12, color='C3')
    ax.set_title(f'k = {k}')
plt.tight_layout(); plt.show()


k = 1 is wild (one outlier carves a region); k = 25 is smooth. Pick k
with cross-validation, not by eye.

## 3. Class imbalance

When one class is rare, **accuracy is a misleading metric**: a model that always predicts the majority class achieves 96.7% accuracy on the Default dataset while catching zero defaulters.

#### Why imbalance hurts

- The model sees far more negatives during training → biased toward predicting negative.
- Standard loss functions treat every sample equally → minority class contributes little to the gradient.
- Cross-validation folds may have no positives if splits are not stratified.

#### Metrics to use on imbalanced data

| Metric | Why it helps |
|---|---|
| Precision / Recall | Focus on the minority class directly |
| F1 / Fβ | Harmonic mean — a model that ignores the minority gets F1 ≈ 0 |
| AUC-ROC | Threshold-free; unaffected by class ratio |
| AUC-PR | Better than AUC-ROC when positive class is < 5% |
| Matthews Correlation Coefficient (MCC) | Balanced metric; works well even at extreme ratios |

Avoid accuracy (and macro accuracy) as the primary metric when classes are imbalanced.

#### Correction strategies

| Strategy | How | When to use |
|---|---|---|
| **Stratified splits** | `train_test_split(stratify=y)` | Always — free and necessary |
| **class_weight='balanced'** | Upweights minority by n / (K · n_k) | Most sklearn classifiers support it |
| **Threshold tuning** | Lower decision threshold from 0.5 | When you need higher recall at cost of precision |
| **Oversampling (SMOTE)** | Synthesise minority samples via interpolation | When you have very few positives |
| **Undersampling** | Randomly drop majority samples | Large datasets where removing data is acceptable |
| **Cost-sensitive learning** | Set misclassification costs asymmetrically | When FN cost ≠ FP cost is known (e.g. medical) |

The minority class matters most in domains like fraud detection, medical diagnosis, and credit risk.


### The majority-class trap

In [ ]:
trivial = np.zeros_like(yte)  # 'always predict No'
print(f'"always No" accuracy = {(trivial == yte).mean():.4f}')
print('… but recall on defaulters = 0. Useless model.')


### Fix: stratify + class_weight

In [ ]:
from sklearn.linear_model import LogisticRegression
# stratify=y preserves class ratio in both train and test splits
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.3, random_state=0, stratify=y)
# class_weight='balanced' upweights minority class inversely to its frequency
lr = LogisticRegression(max_iter=2000, class_weight='balanced').fit(Xtr, ytr)
y_pred = lr.predict(Xte)
print(classification_report(yte, y_pred, target_names=['No', 'Yes']))

Accuracy *drops* but recall on the positive class jumps — almost
always the right trade-off for fraud, default, disease, etc.

### Recap
- Classifier output: a label, often via a probability + threshold.
- Read precision, recall, F1, AUC together. Accuracy alone lies on imbalanced data.
- KNN needs scaling. k controls the bias-variance dial.
- Use `stratify=y` and `class_weight='balanced'` when classes are skewed.

---

# Exercises

Each exercise below is followed by its solution.
Try to solve the tasks yourself before revealing the next cell.

---

## Exercise 1

**Task 1.** Load `penguins`, drop NaN, and fit KNN with `k=5` to
predict `species` from `flipper_length_mm` and `bill_length_mm`.
Use a stratified train/test split and report test accuracy.

In [ ]:
import sys, pathlib
p = pathlib.Path.cwd()
while not (p / 'pyproject.toml').exists() and p != p.parent:
    p = p.parent
if str(p) not in sys.path:
    sys.path.insert(0, str(p))
from shared.data_utils import load_dataset
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
# your code here


### Exercise 1 — Solution

In [ ]:
import sys, pathlib
p = pathlib.Path.cwd()
while not (p / 'pyproject.toml').exists() and p != p.parent:
    p = p.parent
if str(p) not in sys.path:
    sys.path.insert(0, str(p))
from shared.data_utils import load_dataset
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
df = load_dataset('penguins').dropna()
X = df[['flipper_length_mm', 'bill_length_mm']].to_numpy()
y = df['species'].to_numpy()
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.3, random_state=0, stratify=y)
pipe = Pipeline([('s', StandardScaler()), ('knn', KNeighborsClassifier(5))]).fit(Xtr, ytr)
print(f'k=5 test accuracy = {pipe.score(Xte, yte):.4f}')


---

## Exercise 2

**Task 2.** Plot the decision boundary for `k ∈ {1, 5, 25}` on the
(flipper_length_mm, bill_length_mm) plane.

In [ ]:
# your code here


### Exercise 2 — Solution

In [ ]:
import numpy as np, matplotlib.pyplot as plt
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder().fit(y)
y_enc = le.transform(y)
x_min, x_max = X[:,0].min()-2, X[:,0].max()+2
y_min, y_max = X[:,1].min()-2, X[:,1].max()+2
xx, yy = np.meshgrid(np.linspace(x_min, x_max, 120), np.linspace(y_min, y_max, 120))
fig, axes = plt.subplots(1, 3, figsize=(13, 4), sharey=True)
for ax, k in zip(axes, [1, 5, 25]):
    pipe = Pipeline([('s', StandardScaler()), ('knn', KNeighborsClassifier(k))]).fit(X, y_enc)
    Z = pipe.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
    ax.contourf(xx, yy, Z, alpha=0.25, levels=[-0.5,0.5,1.5,2.5])
    for cls in range(3):
        ax.scatter(X[y_enc==cls,0], X[y_enc==cls,1], s=10, label=le.classes_[cls])
    ax.set_title(f'k = {k}'); ax.legend(fontsize=7)
plt.tight_layout(); plt.show()


---

## Exercise 3

**Task 3.** Use 5-fold CV to pick the best `k` from `[1, 3, 5, 11, 21, 51]`.

In [ ]:
# your code here


### Exercise 3 — Solution

In [ ]:
from sklearn.model_selection import cross_val_score
ks = [1, 3, 5, 11, 21, 51]
for k in ks:
    pipe = Pipeline([('s', StandardScaler()), ('knn', KNeighborsClassifier(k))])
    score = cross_val_score(pipe, X, y, cv=5).mean()
    print(f'k={k:3d}  CV accuracy = {score:.4f}')


---

## Exercise 4

**Task 4.** Fit KNN with `k=10` on the Default dataset using `balance` and `income` as features and `default` as target. Compare test accuracy for three distance metrics: Euclidean (`p=2`), Manhattan (`p=1`), and Chebyshev (`metric='chebyshev'`). Print a results table. Use `test_size=0.3, random_state=0, stratify=y`.

In [ ]:
# your code here

### Exercise 4 — Solution

In [ ]:
from shared.data_utils import load_dataset
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
import numpy as np

default4 = load_dataset('default')
d4 = default4.copy()
d4['y'] = (d4['default'] == 'Yes').astype(int)
X4 = d4[['balance', 'income']].to_numpy()
y4 = d4['y'].to_numpy()

# stratify=y preserves class ratio in both train and test splits
Xtr4, Xte4, ytr4, yte4 = train_test_split(X4, y4, test_size=0.3, random_state=0, stratify=y4)

scaler = StandardScaler()
Xtr4s = scaler.fit_transform(Xtr4)
Xte4s = scaler.transform(Xte4)

metrics = [
    ('Euclidean (p=2)',  KNeighborsClassifier(n_neighbors=10, metric='minkowski', p=2)),
    ('Manhattan (p=1)',  KNeighborsClassifier(n_neighbors=10, metric='minkowski', p=1)),
    ('Chebyshev',        KNeighborsClassifier(n_neighbors=10, metric='chebyshev')),
]

print(f"{'Metric':<20}  {'Test Accuracy':>14}")
print('-' * 37)
for name, clf in metrics:
    clf.fit(Xtr4s, ytr4)
    acc = clf.score(Xte4s, yte4)
    print(f"{name:<20}  {acc:>14.4f}")

---

## Exercise 5

**Task 5.** For `k ∈ [1, 5, 15, 51]`, fit KNN on Default (`balance` + `income`). For each k, bin the test set into 10 equally-spaced bins of predicted probability, compute the mean predicted probability and mean actual positive rate per bin, and plot a calibration curve (predicted prob on x, actual freq on y). Which k appears best calibrated?

In [ ]:
# your code here

### Exercise 5 — Solution

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from shared.data_utils import load_dataset
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

default5 = load_dataset('default')
d5 = default5.copy()
d5['y'] = (d5['default'] == 'Yes').astype(int)
X5 = d5[['balance', 'income']].to_numpy()
y5 = d5['y'].to_numpy()

# stratify=y preserves class ratio in both train and test splits
Xtr5, Xte5, ytr5, yte5 = train_test_split(X5, y5, test_size=0.3, random_state=0, stratify=y5)
sc5 = StandardScaler()
Xtr5s = sc5.fit_transform(Xtr5)
Xte5s = sc5.transform(Xte5)

bin_edges = np.linspace(0, 1, 11)  # 10 equally-spaced bins
fig, ax = plt.subplots(figsize=(6, 5))
ax.plot([0, 1], [0, 1], 'k--', alpha=0.4, label='Perfect calibration')

for k in [1, 5, 15, 51]:
    clf = KNeighborsClassifier(n_neighbors=k).fit(Xtr5s, ytr5)
    # predict_proba gives P(positive) — column 1
    probs = clf.predict_proba(Xte5s)[:, 1]
    bin_ids = np.digitize(probs, bin_edges) - 1
    bin_ids = np.clip(bin_ids, 0, 9)
    mean_pred, mean_actual = [], []
    for b in range(10):
        mask = bin_ids == b
        if mask.sum() > 0:
            mean_pred.append(probs[mask].mean())
            mean_actual.append(yte5[mask].mean())
    ax.plot(mean_pred, mean_actual, 'o-', label=f'k={k}')

ax.set_xlabel('Mean predicted probability')
ax.set_ylabel('Actual positive fraction')
ax.set_title('Calibration curves — KNN on Default')
ax.legend(); plt.tight_layout(); plt.show()
print('Larger k (15, 51) tends to be better calibrated: probabilities are averaged over more neighbours.')

---

## Exercise 6

**Task 6.** The Default dataset is heavily imbalanced (~3.3% defaults). Compare three strategies on ROC-AUC using KNN (k=15) on `balance` + `income`:
- **(a)** Vanilla KNN with default threshold (0.5)
- **(b)** KNN with threshold tuning — use `predict_proba` and threshold = 0.1
- **(c)** KNN with SMOTE oversampling (`from imblearn.over_sampling import SMOTE`) — skip gracefully if not installed

Print AUC for each strategy and note how threshold tuning and SMOTE affect recall on the minority class.

In [ ]:
# your code here

### Exercise 6 — Solution

In [ ]:
import numpy as np
from shared.data_utils import load_dataset
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, classification_report

default6 = load_dataset('default')
d6 = default6.copy()
d6['y'] = (d6['default'] == 'Yes').astype(int)
X6 = d6[['balance', 'income']].to_numpy()
y6 = d6['y'].to_numpy()

# stratify=y preserves class ratio in both train and test splits
Xtr6, Xte6, ytr6, yte6 = train_test_split(X6, y6, test_size=0.3, random_state=0, stratify=y6)
sc6 = StandardScaler()
Xtr6s = sc6.fit_transform(Xtr6)
Xte6s = sc6.transform(Xte6)

knn6 = KNeighborsClassifier(n_neighbors=15).fit(Xtr6s, ytr6)
probs6 = knn6.predict_proba(Xte6s)[:, 1]

# (a) Vanilla KNN — default threshold 0.5
y_pred_a = (probs6 >= 0.5).astype(int)
auc_a = roc_auc_score(yte6, probs6)
print(f"(a) Vanilla KNN (threshold=0.5)  ROC-AUC = {auc_a:.4f}")
print(classification_report(yte6, y_pred_a, target_names=['No', 'Yes']))

# (b) Threshold tuning — lower threshold boosts recall on minority class
y_pred_b = (probs6 >= 0.1).astype(int)
auc_b = roc_auc_score(yte6, probs6)  # AUC is threshold-independent
print(f"(b) Threshold tuning (threshold=0.1)  ROC-AUC = {auc_b:.4f}")
print(classification_report(yte6, y_pred_b, target_names=['No', 'Yes']))

# (c) SMOTE — synthesises minority-class examples to balance training set
try:
    from imblearn.over_sampling import SMOTE
    sm = SMOTE(random_state=0)
    Xtr6_sm, ytr6_sm = sm.fit_resample(Xtr6s, ytr6)
    knn6_sm = KNeighborsClassifier(n_neighbors=15).fit(Xtr6_sm, ytr6_sm)
    probs6_sm = knn6_sm.predict_proba(Xte6s)[:, 1]
    y_pred_c = (probs6_sm >= 0.5).astype(int)
    auc_c = roc_auc_score(yte6, probs6_sm)
    print(f"(c) SMOTE + KNN (threshold=0.5)  ROC-AUC = {auc_c:.4f}")
    print(classification_report(yte6, y_pred_c, target_names=['No', 'Yes']))
except ImportError:
    print("(c) SMOTE skipped — install imbalanced-learn: pip install imbalanced-learn")

print("Takeaway: ROC-AUC is the same for (a) and (b) because it is threshold-independent.")
print("Threshold tuning shifts the precision/recall trade-off; SMOTE rebalances the training distribution.")